# Task 4 – Sentiment Analysis

In [1]:
import pandas as pd
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

In [8]:
# Load dataset
sa = pd.read_csv(r"C:\Users\ameya\Dropbox\PC\Downloads\pandas dataset\AmazonFoodReviews.csv")
sa.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


In [9]:
# inspection
print("Rows:", sa.shape[0])
print("Columns:", sa.shape[1])
print("\nMissing values:")
print(sa.isnull().sum())

Rows: 568454
Columns: 10

Missing values:
Id                         0
ProductId                  0
UserId                     0
ProfileName               26
HelpfulnessNumerator       0
HelpfulnessDenominator     0
Score                      0
Time                       0
Summary                   27
Text                       0
dtype: int64


In [10]:
# Keep the main columns
sa = sa[['Text', 'Score']].copy()
sa = sa.dropna()
sa.head()

,Text,Score
0,I have bought several of the Vitality canned d...,5
1,Product arrived labeled as Jumbo Salted Peanut...,1
2,This is a confection that has been around a fe...,4
3,If you are looking for the secret ingredient i...,2
4,Great taffy at a great price. There was a wid...,5


In [11]:
# Convert rating into sentiment
def create_sentiment(score):
    if score <= 2:
        return "Negative"
    elif score == 3:
        return "Neutral"
    else:
        return "Positive"

sa["Sentiment"] = sa["Score"].apply(create_sentiment)
print(sa["Sentiment"].value_counts())

Sentiment
Positive    443777
Negative     82037
Neutral      42640
Name: count, dtype: int64


In [12]:
# text cleaning
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

sa["Clean_Text"] = sa["Text"].apply(clean_text)
sa = sa[sa["Clean_Text"].str.len() > 0]
sa[['Text', 'Clean_Text', 'Sentiment']].head()

,Text,Clean_Text,Sentiment
0,I have bought several of the Vitality canned d...,i have bought several of the vitality canned d...,Positive
1,Product arrived labeled as Jumbo Salted Peanut...,product arrived labeled as jumbo salted peanut...,Negative
2,This is a confection that has been around a fe...,this is a confection that has been around a fe...,Positive
3,If you are looking for the secret ingredient i...,if you are looking for the secret ingredient i...,Negative
4,Great taffy at a great price. There was a wid...,great taffy at a great price there was a wide ...,Positive


In [13]:
# Use a smaller sample for faster processing
sa_sample = sa.sample(min(10000, len(sa)), random_state=42)

X = sa_sample["Clean_Text"]
y = sa_sample["Sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

In [14]:
# Convert text into TF-IDF features
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF features:", X_train_tfidf.shape)

TF-IDF features: (8000, 5000)


In [15]:
# Train a Naive Bayes model
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

predictions = model.predict(X_test_tfidf)

print("Accuracy:", round(accuracy_score(y_test, predictions), 4))

Accuracy: 0.794


In [16]:
# Test a few new reviews
sample_reviews = [
    "This product is excellent and I love it.",
    "The product was okay.",
    "Very disappointing product."
]

sample_cleaned = [clean_text(x) for x in sample_reviews]
sample_features = tfidf.transform(sample_cleaned)
sample_predictions = model.predict(sample_features)

for review, sentiment in zip(sample_reviews, sample_predictions):
    print(f"{sentiment}: {review}")

Positive: This product is excellent and I love it.
Positive: The product was okay.
Negative: Very disappointing product.


## Conclusion

This notebook demonstrates the basic process of sentiment analysis: preparing review data, converting ratings into sentiment labels, cleaning text, creating TF-IDF features, training a Naive Bayes classifier, and making predictions on new reviews.
